# 00 — Dataset Construction

Builds `data/prompts/prompts.csv`: token-aligned minimal pairs for all three conflict families.

**The design rule.** The control arm keeps every clause of the conflict arm and neutralises the
conflict by swapping **one token**. Both arms therefore tokenise to the same length and share the
same two answer tokens. This is what makes activation patching well-defined and makes a
conflict-minus-control difference immune to length and content confounds.

`build_minimal_pair` asserts equal length and an exact differing-token count, so a misaligned pair
cannot enter the dataset at all.

**Ground truth.** There is no `ground_truth` column. `gold_control` is the answer licensed *by
design* in the control arm — written by the generator from the rule text, a committed fact table,
or grammatical gender agreement. The conflict arm has no gold answer; it has a measured outcome.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
%load_ext autoreload
%autoreload 2

In [2]:
from circuit_conflict.utils import load_model
from circuit_conflict import dataset as D
import pandas as pd

model = load_model()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
Loaded GPT-2 Small on mps
  n_layers=12, n_heads=12, d_model=768, d_head=64


## 1. Generate the three categories

In [3]:
frames, rejects = [], []
for fn in (D.build_category_a_df, D.build_category_b_df, D.build_category_c_df):
    df_c, rej = fn(model)
    frames.append(df_c); rejects += rej

df = pd.concat(frames, ignore_index=True)
print(f"{len(df)//2} items / {len(df)} rows built, {len(rejects)} rejected")
for r in rejects[:10]:
    print("  reject:", r)

89 items / 178 rows built, 0 rejected


## 2. Inspect a minimal pair from each category

The two arms should differ in exactly one token.

In [4]:
for cat in ["A", "B", "C"]:
    g = df[df.category == cat]
    c = g[g.arm == "conflict"].iloc[0]; u = g[g.arm == "control"].iloc[0]
    print(f"--- {cat} ---")
    print("  conflict:", c.prompt_text)
    print("  control :", u.prompt_text)
    print(f"  A={c.answer_A!r} (slot-supported)  B={c.answer_B!r}  gold_control={c.gold_control}")
    print(f"  p_slot={c.p_slot}  p_end={c.p_end}  n_tokens={c.n_tokens}\n")

--- A ---
  conflict: When Joseph and Andrew went to the store, he bought a drink. The buyer was
  control : When Sarah and Andrew went to the store, he bought a drink. The buyer was
  A='Joseph' (slot-supported)  B='Andrew'  gold_control=B
  p_slot=2  p_end=17  n_tokens=18

--- B ---
  conflict: Rule one: say south. Rule two: say north. Obeying the rules, I say
  control : Rule one: say south. Rule two: say south. Obeying the rules, I say
  A='north' (slot-supported)  B='south'  gold_control=B
  p_slot=11  p_end=20  n_tokens=21

--- C ---
  conflict: Fact: the capital of Peru is Copenhagen. Question: what is the capital of Peru? Answer: the capital of Peru is
  control : Fact: the capital of Peru is Lima. Question: what is the capital of Peru? Answer: the capital of Peru is
  A='Copenhagen' (slot-supported)  B='Lima'  gold_control=B
  p_slot=8  p_end=25  n_tokens=26



## 3. Preconditions

The one legitimate use of the model at dataset time. Measured on a probe prompt **distinct from
both experimental arms**, gating item *eligibility* rather than the outcome label.

An item is admitted only if the model's preference **reverses** when the single disambiguating
token is swapped. The reversal requirement is what rules out "answer_A is simply the more frequent
word" — the failure mode Category B is most exposed to.

In [5]:
df = D.run_preconditions(model, df)
report = D.precondition_report(df)
print(report.to_string(index=False))

category  n_items  n_pass  pass_rate  median_margin
       A       40      25   0.625000       1.570119
       B       24      23   0.958333       6.936447
       C       25      25   1.000000       6.814991


### The pre-registered Category B gate

This threshold is **specific to Category B**, and it answers one question: GPT-2 Small does not
follow instructions, so are these prompts measuring instruction arbitration at all, or just token
statistics? A pass rate near chance would mean the category is invalid as a test of the mechanism.

It is *not* a general category-validity test. For A and C the same number means something
different — it is the item **admission** rate, i.e. how many generated items showed the effect
strongly enough to be worth measuring. Every admitted item, in every category, individually passed
the reversal criterion.

Reported here for all three so the filtering is visible, but only B is gated on it.

In [6]:
gate = report.set_index("category").pass_rate

# Category B: validity gate (pre-registered threshold 0.70)
verdict = "PASS — valid instruction-arbitration test" if gate["B"] >= 0.70 else \
          "FAIL — report separately as exploratory"
print(f"  B gate: pass rate {gate['B']:.2f}  ->  {verdict}\n")

# A and C: admission rates, reported for transparency (not gated)
for cat in ["A", "C"]:
    print(f"  {cat} admission rate: {gate[cat]:.2f} "
          f"({int(report.set_index('category').n_pass[cat])} items admitted)")

print("\nNote: A required the most filtering. Items were kept only where the model shows the")
print("gender-agreement effect, which is a selection on model behaviour and is stated as a")
print("limitation rather than hidden.")

  B gate: pass rate 0.96  ->  PASS — valid instruction-arbitration test

  A admission rate: 0.62 (25 items admitted)
  C admission rate: 1.00 (25 items admitted)

Note: A required the most filtering. Items were kept only where the model shows the
gender-agreement effect, which is a selection on model behaviour and is stated as a
limitation rather than hidden.


## 4. Save

In [7]:
D.save_prompts(df)
admitted = df[df.passes_precondition]
print(f"admitted {len(admitted)//2} of {len(df)//2} items")

Saved 178 rows -> /Users/acekhan/code/circuit-conflict/data/prompts/prompts.csv
admitted 73 of 89 items
